# Notebook 07 — Gradio Demo

**Features:**
- Toggle RAG on/off
- Toggle between base and fine-tuned model
- Top-K slider
- Retrieved chunks display with source + score
- "Compare 4 configs" tab
- 6 example questions
- `demo.launch(share=True)` for 72-hour public Colab URL

> **Prerequisites:** Notebooks 01–04 must be complete.

In [1]:
from pathlib import Path
import os

if os.path.exists('/content/'):
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/NLP_Final/Source'
else:
    BASE = '../'

print(f'Base directory: {BASE}')

Mounted at /content/drive
Base directory: /content/drive/MyDrive/NLP_Final/Source


In [2]:
%%capture
!pip install -q trl==0.11.4 \
    transformers==4.45.0 peft==0.13.0 accelerate==1.0.1
!pip install -q -U bitsandbytes
!pip install -q faiss-cpu FlagEmbedding==1.2.11
!pip install -q -U "gradio>=5.9.0"
print('Packages installed.')

## 1. Load All Components

In [3]:
import os, json, torch, pickle, inspect
import faiss
from FlagEmbedding import BGEM3FlagModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, LoraConfig

MODEL_ID  = 'Qwen/Qwen2.5-3B-Instruct'
SFT_PATH  = f'{BASE}/models/sft_checkpoint'
INDEX_DIR = f'{BASE}/data/vector_store/faiss_index'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
COMPUTE_DTYPE = (
    torch.bfloat16 if device == 'cuda' and torch.cuda.get_device_properties(0).total_memory > 30e9
    else torch.float16 if device == 'cuda'
    else torch.float32
)
print(f'Device: {device}  |  dtype: {COMPUTE_DTYPE}')

# --- FAISS ---
print('Loading FAISS index...')
faiss_index = faiss.read_index(f'{INDEX_DIR}/index.faiss')
with open(f'{INDEX_DIR}/index.pkl', 'rb') as f:
    index_chunks = pickle.load(f)
print(f'  {faiss_index.ntotal} vectors loaded')

# --- Parent map ---
print('Loading parent chunk map...')
parent_map = {}
parents_path = f'{BASE}/data/chunks/parent_chunks.jsonl'
if os.path.exists(parents_path):
    with open(parents_path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                p = json.loads(line)
                parent_map[p['parent_chunk_id']] = p['text']
    print(f'  {len(parent_map)} parent chunks loaded')
else:
    print(f'  WARNING: parent_chunks.jsonl not found at {parents_path}')

# --- BGE-M3 ---
print('Loading BGE-M3 embedder (CPU)...')
embedder = BGEM3FlagModel('BAAI/bge-m3', use_fp16=False)
print('  BGE-M3 ready (dim=1024)')

# --- Tokenizer & Models ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(SFT_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

IM_END_ID = tokenizer.convert_tokens_to_ids('<|im_end|>')

print('Loading base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto', trust_remote_code=True
)
base_model.config.use_cache = True
base_model.eval()

# Patch adapter_config: strip keys unknown to peft==0.13.0
_cfg_path = f'{SFT_PATH}/adapter_config.json'
_valid_keys = set(inspect.signature(LoraConfig.__init__).parameters) - {'self'}
with open(_cfg_path) as f:
    _cfg = json.load(f)
_removed = [k for k in list(_cfg) if k not in _valid_keys]
for k in _removed:
    _cfg.pop(k)
with open(_cfg_path, 'w') as f:
    json.dump(_cfg, f, indent=2)
if _removed:
    print(f'  adapter_config patched — removed: {_removed}')

print('Loading SFT adapter...')
ft_model = PeftModel.from_pretrained(base_model, SFT_PATH)
ft_model.eval()

print('All components loaded.')

Device: cuda  |  dtype: torch.float16
Loading FAISS index...
  280 vectors loaded
Loading parent chunk map...
  232 parent chunks loaded
Loading BGE-M3 embedder (CPU)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

bm25.jpg:   0%|          | 0.00/132k [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading SFT adapter...
All components loaded.


In [4]:
print('Merging LoRA weights...')
ft_model = ft_model.merge_and_unload()  # loại bỏ overhead adapter
ft_model.eval()

Merging LoRA weights...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:336: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-0

In [5]:
SYSTEM_PROMPT = (
    'Ban ten la TDTU slay - mot tro ly AI cua Truong Dai hoc Ton Duc Thang (TDTU), '
    'duoc xay dung de ho tro sinh vien va giang vien tra loi cac cau hoi '
    've quy che, quy dinh, chinh sach cua truong. Tra loi bang tieng Viet.\n\n'
    'NGUYEN TAC:\n'
    '- Chi tra loi dua tren THONG TIN THAM KHAO duoc cung cap phia duoi\n'
    '- Neu khong co thong tin tham khao hoac thong tin khong lien quan den cau hoi, '
    'hay tra loi: "Toi khong co thong tin nay trong tai lieu. '
    'Ban co the lien he Phong Dao tao TDTU de duoc ho tro."\n'
    '- Neu duoc hoi ve ban than (Vi du: Ban la ai?), hay tra loi ngan gon rang '
    'ban la tro ly AI cua TDTU ho tro tu van quy che va dung lai\n'
    '- Khong tu y bịa dat hoac suy dien ngoai tai lieu\n'
    '- Giu giong dieu than thien, chuyen nghiep'
)

## 2. Core Inference Functions

In [6]:
import numpy as np
from transformers import TextIteratorStreamer, StoppingCriteria, StoppingCriteriaList
import threading

_CONTEXT_WORDS = {'thế', 'vậy', 'còn', 'cũng', 'đó', 'này', 'kia', 'ấy'}

RETRIEVAL_THRESHOLD = 0.5

def retrieve(query: str, k: int = 5) -> list[dict]:
    out   = embedder.encode([query], return_dense=True, return_sparse=False, return_colbert_vecs=False)
    q_vec = np.array(out['dense_vecs'], dtype='float32')
    scores, indices = faiss_index.search(q_vec, k)
    return [
        {'chunk': index_chunks[i], 'score': float(scores[0][j])}
        for j, i in enumerate(indices[0])
        if i < len(index_chunks) and float(scores[0][j]) >= RETRIEVAL_THRESHOLD
    ]


def get_parent_texts(results: list[dict], top_n: int = 3) -> list[str]:
    """Parent-child strategy: embed child → look up and inject parent text."""
    seen, texts = set(), []
    for r in results:
        pid = r['chunk'].get('parent_chunk_id', '')
        if pid in seen:
            continue
        seen.add(pid)
        texts.append(parent_map.get(pid) or r['chunk']['text'])
        if len(texts) >= top_n:
            break
    return texts


def enhance_query(question: str, history: list[dict]) -> str:
    """Expand query with previous user turn for context-dependent follow-ups."""
    words = set(question.lower().split())
    if not (words & _CONTEXT_WORDS):
        return question
    prev = next((t['content'] for t in reversed(history) if t['role'] == 'user'), None)
    return f'{prev} {question}' if prev else question


def build_prompt(question: str, context_texts: list[str] | None, history: list[dict]) -> str:
    """Build Qwen2.5 ChatML prompt with history + optional RAG context."""
    prompt = f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
    if context_texts:
        ctx_str = '\n\n'.join(f'[{i+1}] {t}' for i, t in enumerate(context_texts))
        prompt += f'<|im_start|>system\nTHONG TIN THAM KHAO TU QUY CHE TDTU:\n{ctx_str}<|im_end|>\n'
    for turn in history[-12:]:   # tối đa 6 lượt (12 messages)
        prompt += f'<|im_start|>{turn["role"]}\n{turn["content"]}<|im_end|>\n'
    prompt += f'<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n'
    return prompt


def generate(model, prompt: str, max_new_tokens: int = 400) -> str:
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=IM_END_ID,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    if '<|im_end|>' in response:
        response = response.split('<|im_end|>')[0].strip()
    return response

_stop_event = threading.Event()

class _StopOnEvent(StoppingCriteria):
    def __call__(self, input_ids, scores, **kwargs):
        return _stop_event.is_set()

def generate_stream(model, prompt: str, max_new_tokens: int = 256):
    _stop_event.clear()
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1536).to(model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=5.0)
    gen_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=IM_END_ID,
        stopping_criteria=StoppingCriteriaList([_StopOnEvent()]),
    )
    threading.Thread(target=model.generate, kwargs=gen_kwargs, daemon=True).start()
    result = ''
    try:
        for token in streamer:
            if '<|im_end|>' in token:
                break
            result += token
            yield result
    except GeneratorExit:
        _stop_event.set()   # báo cho generate thread dừng sớm
        raise

print('Inference functions ready.')

Inference functions ready.


## 3. Gradio Interface

In [7]:
import gradio as gr

VIDEO_URL = "drive.google.com/..."   # ← thay bằng link video demo thực

CSS = """
.main-header { background:#1a2744; color:white; padding:16px 24px;
               border-radius:10px; margin-bottom:10px; }
.panel-left   { background:#1a2744; color:white; text-align:center;
               padding:10px 16px; border-radius:8px 8px 0 0;
               font-weight:bold; font-size:1.05em; letter-spacing:.5px; }
.panel-right  { background:#c0392b; color:white; text-align:center;
               padding:10px 16px; border-radius:8px 8px 0 0;
               font-weight:bold; font-size:1.05em; letter-spacing:.5px; }
.footer-bar   { background:#1a2744; color:white; text-align:center;
               padding:12px; border-radius:8px; margin-top:10px; }
#ans-out textarea { background:#e8f5e9 !important; border:2px solid #27ae60 !important; }
"""

# ── Helper: HTML card cho config comparison ───────────────────────────────────

def _card(label: str, color: str, title: str, desc: str) -> str:
    short = desc[:800] + '…' if len(desc) > 800 else desc
    return (
        f'<div style="display:flex;align-items:center;gap:14px;'
        f'background:white;border-radius:8px;padding:12px;margin:8px 0;'
        f'box-shadow:0 1px 4px rgba(0,0,0,.12);">'
        f'<div style="width:44px;height:44px;border-radius:50%;background:{color};'
        f'display:flex;align-items:center;justify-content:center;'
        f'color:white;font-weight:bold;font-size:1.2em;flex-shrink:0;">{label}</div>'
        f'<div><div style="font-weight:bold;margin-bottom:2px;">{title}</div>'
        f'<div style="color:#555;font-size:.88em;">{short}</div></div></div>'
    )

_PLACEHOLDER = (
    _card('A','#7f8c8d','Base, no RAG',   'Trả lời chung chung, có thể bịa thông tin…'),
    _card('B','#2980b9','Base + RAG',     'Có căn cứ từ văn bản, văn phong tự nhiên…'),
    _card('C','#f39c12','FT, no RAG',     'Văn phong TDTU nhưng thiếu chi tiết cụ thể…'),
    _card('D','#27ae60','FT + RAG ⭐',   'Văn phong TDTU + căn cứ rõ ràng → BEST…'),
)

# ── Backend functions ─────────────────────────────────────────────────────────

def run_qa(question: str, use_rag: bool, use_ft: bool, top_k: int):
    if not question.strip():
        return '', ''
    model = ft_model if use_ft else base_model
    results, parent_texts = [], []
    if use_rag:
        results      = retrieve(question, k=int(top_k))
        parent_texts = get_parent_texts(results, top_n=3)
    prompt = build_prompt(question, parent_texts if use_rag else None, history=[])
    answer = generate(model, prompt, max_new_tokens=400)

    chunks_info = ''
    if results:
        src  = results[0]['chunk'].get('source_file', '').replace(' ', '_')
        nums = ', '.join(f'#{r["chunk"].get("chunk_id","?")[-4:]}' for r in results[:3])
        chunks_info = f'📎 Retrieved chunks (top-{min(3,len(results))}): {src}  {nums}'
    return answer, chunks_info

def run_qa_stream(question: str, use_rag: bool, use_ft: bool, top_k: int):
    if not question.strip():
        yield '', ''
        return

    model = ft_model if use_ft else base_model
    results, parent_texts = [], []
    if use_rag:
        results      = retrieve(question, k=int(top_k))
        parent_texts = get_parent_texts(results, top_n=3)

    # chunks info (giống run_qa cũ)
    chunks_info = ''
    if results:
        src  = results[0]['chunk'].get('source_file', '').replace(' ', '_')
        nums = ', '.join(f'#{r["chunk"].get("chunk_id","?")[-4:]}' for r in results[:3])
        chunks_info = f'📎 Retrieved chunks (top-{min(3,len(results))}): {src}  {nums}'

    prompt = build_prompt(question, parent_texts if use_rag else None, history=[])

    # Stream từng token ra UI
    for partial_answer in generate_stream(model, prompt):
        yield partial_answer, chunks_info

def run_compare(question: str):
    if not question.strip():
        yield _PLACEHOLDER
        return

    # Hiện loading ngay trong lúc retrieve (vài giây)
    ld_a = _card('A', '#7f8c8d', 'Base, no RAG', '⏳ Đang tạo…')
    ld_b = _card('B', '#2980b9', 'Base + RAG',   '⏳ Đang tạo…')
    ld_c = _card('C', '#f39c12', 'FT, no RAG',   '⏳ Đang tạo…')
    ld_d = _card('D', '#27ae60', 'FT + RAG ⭐',  '⏳ Đang tạo…')
    yield ld_a, ld_b, ld_c, ld_d

    results      = retrieve(question, k=5)
    parent_texts = get_parent_texts(results, top_n=3)

    # A — Base, no RAG
    ans_a = ''
    for ans_a in generate_stream(base_model, build_prompt(question, None, history=[]), max_new_tokens=256):
        yield _card('A', '#7f8c8d', 'Base, no RAG', ans_a), ld_b, ld_c, ld_d

    # B — Base + RAG
    ans_b = ''
    for ans_b in generate_stream(base_model, build_prompt(question, parent_texts, history=[]), max_new_tokens=256):
        yield _card('A', '#7f8c8d', 'Base, no RAG', ans_a), _card('B', '#2980b9', 'Base + RAG', ans_b), ld_c, ld_d

    # C — FT, no RAG
    ans_c = ''
    for ans_c in generate_stream(ft_model, build_prompt(question, None, history=[]), max_new_tokens=256):
        yield _card('A', '#7f8c8d', 'Base, no RAG', ans_a), _card('B', '#2980b9', 'Base + RAG', ans_b), _card('C', '#f39c12', 'FT, no RAG', ans_c), ld_d

    # D — FT + RAG
    ans_d = ''
    for ans_d in generate_stream(ft_model, build_prompt(question, parent_texts, history=[]), max_new_tokens=256):
        yield _card('A', '#7f8c8d', 'Base, no RAG', ans_a), _card('B', '#2980b9', 'Base + RAG', ans_b), _card('C', '#f39c12', 'FT, no RAG', ans_c), _card('D', '#27ae60', 'FT + RAG ⭐', ans_d)

# ── UI ────────────────────────────────────────────────────────────────────────

with gr.Blocks(css=CSS, title='TDTU QA Demo') as demo:

    gr.HTML("""
    <div class="main-header">
      <div style="font-size:1.45em;font-weight:bold;">
        DEMO HỆ THỐNG — GIAO DIỆN GRADIO
      </div>
      <div style="opacity:.7;margin-top:4px;">
        Chương 6.1 — Triển khai web app trên Colab với share=True
      </div>
    </div>""")

    with gr.Row(equal_height=False):

        # ── LEFT: Q&A ──────────────────────────────────────────────────────
        with gr.Column():
            gr.HTML('<div class="panel-left">🔍&nbsp; TAB 1 — HỎI ĐÁP</div>')
            q_box = gr.Textbox(label='Câu hỏi:', lines=2,
                               placeholder='Nhập câu hỏi về quy chế TDTU…')
            with gr.Row():
                rag_cb = gr.Checkbox(label='RAG ON',      value=True)
                ft_cb  = gr.Checkbox(label='Fine-tuned',  value=True)
                topk   = gr.Slider(1, 10, value=5, step=1, label='Top-K')
            with gr.Row():
                send_btn = gr.Button('Gửi câu hỏi',   variant='primary')
                stop_btn = gr.Button('Dừng ⛔',         variant='stop')
                cmp_btn  = gr.Button('So sánh A/B/C/D', variant='secondary')
            ans_out    = gr.Textbox(label='Trả lời:', lines=7,
                                    interactive=False, elem_id='ans-out')
            chunks_md  = gr.Markdown('')

        # ── RIGHT: Compare ─────────────────────────────────────────────────
        with gr.Column():
            gr.HTML('<div class="panel-right">🏆&nbsp; TAB 2 — SO SÁNH 4 CẤU HÌNH</div>')
            cfg_a = gr.HTML(_PLACEHOLDER[0])
            cfg_b = gr.HTML(_PLACEHOLDER[1])
            cfg_c = gr.HTML(_PLACEHOLDER[2])
            cfg_d = gr.HTML(_PLACEHOLDER[3])

    gr.HTML(f'<div class="footer-bar">▶ Video demo 3–5 phút tại: {VIDEO_URL}</div>')

    # Events
    send_btn.click(fn=run_qa_stream,
                   inputs=[q_box, rag_cb, ft_cb, topk],
                   outputs=[ans_out, chunks_md])

    cmp_btn.click(fn=run_compare,
                  inputs=[q_box],
                  outputs=[cfg_a, cfg_b, cfg_c, cfg_d])

    send_event = send_btn.click(
    fn=run_qa_stream,
    inputs=[q_box, rag_cb, ft_cb, topk],
    outputs=[ans_out, chunks_md],
    )
    cmp_event = cmp_btn.click(
        fn=run_compare,
        inputs=[q_box],
        outputs=[cfg_a, cfg_b, cfg_c, cfg_d],
    )
    # Dừng cả 2 luồng bằng 1 nút
    stop_btn.click(fn=None, cancels=[send_event, cmp_event])
print('Gradio interface built.')

/tmp/ipykernel_494/1457271572.py:122: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, title='TDTU QA Demo') as demo:


Gradio interface built.


In [8]:
  # theme chuyển sang launch() từ Gradio 6.x
demo.launch(share=True, debug=False, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://42ab26498ba9d4558d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
